# LeetCode 408 — Valid Word Abbreviation bằng Python

**Link đề:** https://leetcode.com/problems/valid-word-abbreviation/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def validWordAbbreviation(self, word: str, abbr: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu luật **viết tắt từ** (abbreviation): thay các đoạn con không kề nhau bằng độ dài của chúng, không có số 0 dẫn đầu.
2. Nắm kỹ thuật **hai con trỏ** (two pointers) đi song song trên hai chuỗi, và cách **đọc một số nhiều chữ số** từ chuỗi (`num = num * 10 + digit`).
3. Biết `str.isdigit()`, `ord()`, so sánh ký tự, và cách xử lý trường hợp số `0` dẫn đầu.
4. Viết 4 lời giải: brute force sinh mọi viết tắt (chỉ để kiểm tra) → tách token rồi so khớp → hai con trỏ (chính) → biểu thức chính quy `re`.
5. Trace từng bước, chứng minh, Time/Space, lỗi hay gặp, biến thể.

## 1. Đề bài

Một chuỗi có thể được **viết tắt** bằng cách thay **một số đoạn con không rỗng, không kề nhau** bằng **độ dài** của chúng. Độ dài **không được có số 0 dẫn đầu**.

Ví dụ `"substitution"` có thể viết tắt thành (không giới hạn):

| Viết tắt | Cách tách | Hợp lệ? |
|---|---|:---:|
| `"s10n"` | `s` + *ubstitutio* (10) + `n` | ✅ |
| `"sub4u4"` | `sub` + *stit* (4) + `u` + *tion* (4) | ✅ |
| `"12"` | *substitution* (12) | ✅ |
| `"su3i1u2on"` | `su` + *bst* + `i` + *t* + `u` + *ti* + `on` | ✅ |
| `"substitution"` | không thay gì | ✅ |
| `"s55n"` | hai đoạn thay bị **kề nhau** | ❌ |
| `"s010n"` | số có **0 dẫn đầu** | ❌ |
| `"s0ubstitution"` | thay một đoạn **rỗng** | ❌ |

Cho chuỗi `word` và chuỗi viết tắt `abbr`, trả về `True` nếu `abbr` là một viết tắt hợp lệ của `word`.

**Ví dụ:**
- `word = "internationalization"`, `abbr = "i12iz4n"` → `True` (`i` + 12 ký tự + `iz` + 4 ký tự + `n`).
- `word = "apple"`, `abbr = "a2e"` → `False` (`a` + 2 ký tự → tới `l`, nhưng `abbr` đòi `e`).

**Constraints:**
- `1 <= word.length <= 20`, `word` chỉ gồm chữ thường.
- `1 <= abbr.length <= 10`, `abbr` gồm chữ thường và chữ số.
- Mọi số nguyên trong `abbr` vừa kiểu 32-bit.

### 1.1. Đọc `abbr` như một "chỉ dẫn đi bộ"

Hãy tưởng tượng ta đi trên `word` bằng một ngón tay:
- Gặp **chữ cái** trong `abbr` → ký tự hiện tại của `word` phải **giống hệt**, rồi bước 1.
- Gặp **số** `k` trong `abbr` → **nhảy** qua `k` ký tự của `word`.
- Cuối cùng, ngón tay phải dừng **đúng** ở cuối `word` và `abbr` cũng hết.

Quan sát quan trọng về `"s55n"`: khi đọc số ta đọc **tất cả chữ số liền nhau** → đó là số **55**, không phải 5 và 5. Vì thế luật "không kề nhau" tự động được đảm bảo.

## 2. Nền tảng Python

### 2.1. Duyệt chuỗi bằng chỉ số và `str.isdigit()`

`s[i]` là ký tự ở vị trí `i` (bắt đầu từ 0). `ch.isdigit()` trả `True` nếu `ch` là chữ số `'0'..'9'`.

In [1]:
abbr = "i12iz4n"
for i in range(len(abbr)):
    ch = abbr[i]
    print(i, repr(ch), "chữ số" if ch.isdigit() else "chữ cái")

0 'i' chữ cái
1 '1' chữ số
2 '2' chữ số
3 'i' chữ cái
4 'z' chữ cái
5 '4' chữ số
6 'n' chữ cái


### 2.2. Đổi ký tự chữ số thành số: `int(ch)` hoặc `ord(ch) - ord('0')`

`ord(ch)` trả mã Unicode của ký tự. Các chữ số `'0'..'9'` có mã liên tiếp `48..57`, nên `ord(ch) - ord('0')` cho đúng giá trị chữ số.

In [2]:
for ch in "0179":
    print(ch, "-> ord:", ord(ch), "| ord-ord('0'):", ord(ch) - ord("0"), "| int:", int(ch))

0 -> ord: 48 | ord-ord('0'): 0 | int: 0
1 -> ord: 49 | ord-ord('0'): 1 | int: 1
7 -> ord: 55 | ord-ord('0'): 7 | int: 7
9 -> ord: 57 | ord-ord('0'): 9 | int: 9


### 2.3. Đọc một số nhiều chữ số từ trái sang phải

Công thức "dồn chữ số": `num = num * 10 + digit`.

Ví dụ đọc `"12"`: `num = 0 → 0*10+1 = 1 → 1*10+2 = 12`. Đây là cách đọc số **mà không cắt chuỗi**.

In [3]:
s = "x123y"
i = 1
num = 0
while i < len(s) and s[i].isdigit():
    num = num * 10 + int(s[i])
    print(f"  đọc '{s[i]}' -> num = {num}")
    i += 1
print("số đọc được:", num, "| i dừng tại", i, "ký tự", repr(s[i]))

  đọc '1' -> num = 1
  đọc '2' -> num = 12
  đọc '3' -> num = 123
số đọc được: 123 | i dừng tại 4 ký tự 'y'


### 2.4. Hai con trỏ (two pointers)

Dùng **hai chỉ số** `i` (trên `word`) và `j` (trên `abbr`), mỗi chỉ số tiến theo nhịp riêng:
- chữ cái: cả `i` và `j` cùng +1,
- số `k`: `j` tiến qua các chữ số, còn `i` tiến `k` bước.

Ví dụ nhỏ trước: so hai chuỗi bằng hai con trỏ (giống `==`).

In [4]:
def same_by_two_pointers(a: str, b: str) -> bool:
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] != b[j]:
            return False
        i += 1
        j += 1
    return i == len(a) and j == len(b)     # cả hai phải hết cùng lúc

print(same_by_two_pointers("abc", "abc"), same_by_two_pointers("abc", "ab"), same_by_two_pointers("abc", "abd"))

True False False


## 3. Cách trực giác (baseline) — sinh mọi viết tắt của `word`

Cách "ngây thơ": liệt kê **tất cả** viết tắt hợp lệ của `word` rồi xem `abbr` có trong đó không.

Mỗi ký tự của `word` hoặc **giữ lại**, hoặc **bị thay**; các ký tự bị thay liền nhau gộp thành một số. Có `2^n` cách chọn → với `n = 20` là hơn 1 triệu chuỗi: chạy được nhưng rất phí. Ta dùng nó làm **oracle** cho từ ngắn.

Kỹ thuật: dùng **bitmask** — số `mask` từ `0` đến `2^n - 1`, bit thứ `i` bằng 1 nghĩa là ký tự `i` bị thay.

In [5]:
def all_abbreviations(word: str) -> set:
    n = len(word)
    result = set()
    for mask in range(1 << n):              # 1 << n = 2^n
        parts = []
        count = 0                           # số ký tự bị thay đang dồn
        for i in range(n):
            if mask >> i & 1:               # bit i bật -> thay ký tự i
                count += 1
            else:
                if count:
                    parts.append(str(count))
                    count = 0
                parts.append(word[i])
        if count:
            parts.append(str(count))
        result.add("".join(parts))
    return result

abbrs = all_abbreviations("word")
print(len(abbrs), sorted(abbrs))

16 ['1o1d', '1o2', '1or1', '1ord', '2r1', '2rd', '3d', '4', 'w1r1', 'w1rd', 'w2d', 'w3', 'wo1d', 'wo2', 'wor1', 'word']


In [6]:
def valid_abbr_bruteforce(word: str, abbr: str) -> bool:
    return abbr in all_abbreviations(word)

print(valid_abbr_bruteforce("apple", "a3e"))    # True
print(valid_abbr_bruteforce("apple", "a2e"))    # False
print(valid_abbr_bruteforce("apple", "a03e"))   # False: số 0 dẫn đầu không bao giờ được sinh ra

True
False
False


Cú pháp mới: `1 << n` là dịch trái bit, bằng `2**n`. `mask >> i & 1` lấy bit thứ `i` của `mask` (`>>` ưu tiên hơn `&`). Brute force này **không bao giờ** sinh số có `0` dẫn đầu hay số `0`, nên nó là chuẩn tham chiếu tốt.

## 4. Nút thắt và insight

**Nút thắt:** sinh `2^n` chuỗi chỉ để kiểm tra **một** chuỗi — lãng phí khủng khiếp. Ta không cần biết mọi viết tắt; chỉ cần **mô phỏng** `abbr` trên `word` đúng một lần.

**Insight:** đọc `abbr` từ trái sang phải với hai con trỏ:
1. `abbr[j]` là chữ cái → so với `word[i]`; khác → `False`; giống → `i += 1, j += 1`.
2. `abbr[j]` là chữ số → nếu là `'0'` → `False` (0 dẫn đầu hoặc đoạn rỗng). Ngược lại đọc trọn số `k`, `i += k`.
3. Nếu `i` vượt `len(word)` → `False`.
4. Kết thúc: hợp lệ khi `i == len(word)` **và** `j == len(abbr)`.

In [7]:
# Mô phỏng tay "i12iz4n" trên "internationalization"
word = "internationalization"
print(len(word))
print("sau 'i' + 12 bước, i =", 1 + 12, "-> word[13] =", word[13])
print("sau 'iz' i =", 15, "+ 4 =", 19, "-> word[19] =", word[19])

20
sau 'i' + 12 bước, i = 13 -> word[13] = i
sau 'iz' i = 15 + 4 = 19 -> word[19] = n


## 5. Lời giải chính — hai con trỏ

**Đoạn 1 — khởi tạo:** `i, j = 0, 0`; `n, m = len(word), len(abbr)`.

**Đoạn 2 — vòng lặp chính** `while i < n and j < m`:
```python
if abbr[j].isdigit():
    if abbr[j] == "0":          # chặn "0", "01", "s0ubst..."
        return False
    num = 0
    while j < m and abbr[j].isdigit():
        num = num * 10 + int(abbr[j])
        j += 1
    i += num
else:
    if word[i] != abbr[j]:
        return False
    i += 1
    j += 1
```

**Đoạn 3 — kết luận:** `return i == n and j == m`.

Nếu `i` nhảy quá `n` thì vòng lặp dừng và `i == n` sai → `False`. Không cần kiểm tra riêng.

In [8]:
def valid_abbr_two_pointers(word: str, abbr: str) -> bool:
    i, j = 0, 0
    n, m = len(word), len(abbr)
    while i < n and j < m:
        if abbr[j].isdigit():
            if abbr[j] == "0":
                return False
            num = 0
            while j < m and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])
                j += 1
            i += num
        else:
            if word[i] != abbr[j]:
                return False
            i += 1
            j += 1
    return i == n and j == m


print(valid_abbr_two_pointers("internationalization", "i12iz4n"))  # True
print(valid_abbr_two_pointers("apple", "a2e"))                     # False
print(valid_abbr_two_pointers("substitution", "s010n"))            # False

True
False
False


## 6. TRACE từng bước

Bảng trace `word = "internationalization"`, `abbr = "i12iz4n"`:

| Bước | j | abbr[j] | Loại | Hành động | i sau |
|:---:|:---:|:---:|---|---|:---:|
| 1 | 0 | `i` | chữ | `word[0]='i'` khớp | 1 |
| 2 | 1–2 | `12` | số | nhảy 12 | 13 |
| 3 | 3 | `i` | chữ | `word[13]='i'` khớp | 14 |
| 4 | 4 | `z` | chữ | `word[14]='z'` khớp | 15 |
| 5 | 5 | `4` | số | nhảy 4 | 19 |
| 6 | 6 | `n` | chữ | `word[19]='n'` khớp | 20 |

Kết thúc `i = 20 = n`, `j = 7 = m` → `True`.

In [9]:
def trace_valid_abbr(word: str, abbr: str) -> bool:
    i, j = 0, 0
    n, m = len(word), len(abbr)
    print(f"word={word!r} (n={n}), abbr={abbr!r} (m={m})")
    while i < n and j < m:
        if abbr[j].isdigit():
            if abbr[j] == "0":
                print(f"  j={j}: gặp '0' dẫn đầu -> False")
                return False
            start = j
            num = 0
            while j < m and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])
                j += 1
            print(f"  j={start}..{j-1}: số {num} -> i: {i} -> {i + num}")
            i += num
        else:
            ok = word[i] == abbr[j]
            print(f"  j={j}: chữ '{abbr[j]}' vs word[{i}]='{word[i]}' -> {'khớp' if ok else 'KHÁC'}")
            if not ok:
                return False
            i += 1
            j += 1
    res = i == n and j == m
    print(f"  dừng: i={i}, j={j} -> i==n? {i == n}, j==m? {j == m} -> {res}")
    return res

trace_valid_abbr("internationalization", "i12iz4n")

word='internationalization' (n=20), abbr='i12iz4n' (m=7)
  j=0: chữ 'i' vs word[0]='i' -> khớp
  j=1..2: số 12 -> i: 1 -> 13
  j=3: chữ 'i' vs word[13]='i' -> khớp
  j=4: chữ 'z' vs word[14]='z' -> khớp
  j=5..5: số 4 -> i: 15 -> 19
  j=6: chữ 'n' vs word[19]='n' -> khớp
  dừng: i=20, j=7 -> i==n? True, j==m? True -> True


True

### 6.1. Trace các nhánh thất bại

- `"apple"`, `"a2e"`: khớp `a`, nhảy 2 tới `i=3` (`'l'`), so với `'e'` → khác.
- `"apple"`, `"a5"`: nhảy quá đuôi (`i = 6 > 5`) → `i == n` sai.
- `"apple"`, `"apple1"`: khớp hết 5 chữ, `i = 5 = n` nên vòng lặp dừng **trước** khi đọc `'1'`; lúc đó `j = 5 < m = 6` → `j == m` sai → `False`.

In [10]:
trace_valid_abbr("apple", "a2e")
print()
trace_valid_abbr("apple", "a5")
print()
trace_valid_abbr("apple", "apple1")

word='apple' (n=5), abbr='a2e' (m=3)
  j=0: chữ 'a' vs word[0]='a' -> khớp
  j=1..1: số 2 -> i: 1 -> 3
  j=2: chữ 'e' vs word[3]='l' -> KHÁC

word='apple' (n=5), abbr='a5' (m=2)
  j=0: chữ 'a' vs word[0]='a' -> khớp
  j=1..1: số 5 -> i: 1 -> 6
  dừng: i=6, j=2 -> i==n? False, j==m? True -> False

word='apple' (n=5), abbr='apple1' (m=6)
  j=0: chữ 'a' vs word[0]='a' -> khớp
  j=1: chữ 'p' vs word[1]='p' -> khớp
  j=2: chữ 'p' vs word[2]='p' -> khớp
  j=3: chữ 'l' vs word[3]='l' -> khớp
  j=4: chữ 'e' vs word[4]='e' -> khớp
  dừng: i=5, j=5 -> i==n? True, j==m? False -> False


False

## 7. Các lời giải khác

### 7.1. Cách 3 — tách `abbr` thành token rồi so khớp

Tách `abbr` thành danh sách token: chữ cái riêng lẻ hoặc cả cụm số. Ví dụ `"i12iz4n"` → `['i', '12', 'i', 'z', '4', 'n']`. Sau đó duyệt token. Cách này tách **phân tích cú pháp** (parsing) khỏi **so khớp**, dễ đọc với người mới.

In [11]:
def tokenize(abbr: str) -> list:
    tokens = []
    j = 0
    while j < len(abbr):
        if abbr[j].isdigit():
            start = j
            while j < len(abbr) and abbr[j].isdigit():
                j += 1
            tokens.append(abbr[start:j])        # slicing lấy cả cụm số
        else:
            tokens.append(abbr[j])
            j += 1
    return tokens


def valid_abbr_tokens(word: str, abbr: str) -> bool:
    i = 0
    for tok in tokenize(abbr):
        if tok.isdigit():
            if tok[0] == "0":
                return False
            i += int(tok)
        else:
            if i >= len(word) or word[i] != tok:
                return False
            i += 1
    return i == len(word)


print(tokenize("i12iz4n"), tokenize("s010n"))
print(valid_abbr_tokens("internationalization", "i12iz4n"), valid_abbr_tokens("apple", "a2e"))

['i', '12', 'i', 'z', '4', 'n'] ['s', '010', 'n']
True False


Chú ý `abbr[start:j]` là **slicing**: lấy các ký tự từ `start` đến `j - 1`. Ở cách này phải tự kiểm tra `i >= len(word)` trước khi đọc `word[i]` (vì không còn điều kiện `i < n` của `while`).

### 7.2. Cách 4 — biểu thức chính quy (regular expression)

Module chuẩn `re` có thể biến `abbr` thành một **mẫu** (pattern): mỗi số `k` thành `.{k}` (đúng `k` ký tự bất kỳ), chữ cái giữ nguyên. Rồi kiểm tra `word` **khớp toàn bộ** mẫu bằng `re.fullmatch`.

Phải chặn số có `0` dẫn đầu trước bằng regex `(?<!\d)0` ("chữ số 0 mà đứng trước nó không phải chữ số").

In [12]:
import re

print(re.findall(r"\d+|[a-z]", "i12iz4n"))      # \d+ = một hoặc nhiều chữ số
print(re.fullmatch(r"i.{12}iz.{4}n", "internationalization") is not None)
print(re.search(r"(?<!\d)0", "s010n") is not None, re.search(r"(?<!\d)0", "s10n") is not None)

['i', '12', 'i', 'z', '4', 'n']
True
True False


In [13]:
def valid_abbr_regex(word: str, abbr: str) -> bool:
    if re.search(r"(?<!\d)0", abbr):
        return False
    pattern = re.sub(r"\d+", lambda mt: ".{" + mt.group() + "}", abbr)
    return re.fullmatch(pattern, word) is not None

print(re.sub(r"\d+", lambda mt: ".{" + mt.group() + "}", "i12iz4n"))
print(valid_abbr_regex("internationalization", "i12iz4n"), valid_abbr_regex("apple", "a2e"))

i.{12}iz.{4}n
True False


`lambda mt: ...` là **hàm vô danh** một dòng; `re.sub` gọi nó cho mỗi cụm số tìm được (`mt.group()` là chuỗi số). Cách regex ngắn nhưng khó đọc và khó debug — không nên là lựa chọn đầu tiên khi phỏng vấn.

## 8. Code nộp LeetCode

Bản nộp là cách hai con trỏ: `O(m)` thời gian, `O(1)` bộ nhớ, không cần import.

In [14]:
class Solution:
    def validWordAbbreviation(self, word: str, abbr: str) -> bool:
        i = j = 0
        n, m = len(word), len(abbr)
        while i < n and j < m:
            if abbr[j].isdigit():
                if abbr[j] == "0":
                    return False
                num = 0
                while j < m and abbr[j].isdigit():
                    num = num * 10 + int(abbr[j])
                    j += 1
                i += num
            else:
                if word[i] != abbr[j]:
                    return False
                i += 1
                j += 1
        return i == n and j == m


sol = Solution()
print(sol.validWordAbbreviation("internationalization", "i12iz4n"))  # True
print(sol.validWordAbbreviation("apple", "a2e"))                     # False

True
False


## 9. Vì sao đúng? (invariant)

**Bất biến:** ở đầu mỗi vòng lặp, tiền tố `abbr[:j]` là một viết tắt hợp lệ của tiền tố `word[:i]`, và `abbr[:j]` không kết thúc giữa một cụm số.

- Ban đầu `i = j = 0`: chuỗi rỗng viết tắt chuỗi rỗng → đúng.
- Nhánh chữ cái: thêm cùng một ký tự vào cả hai tiền tố → vẫn hợp lệ.
- Nhánh số: đọc **trọn** cụm số `k` (không 0 dẫn đầu, nên `k >= 1` — đoạn thay không rỗng), thay `k` ký tự kế tiếp → vẫn hợp lệ. Vì ta luôn đọc trọn cụm số, hai đoạn thay không bao giờ kề nhau.
- Mọi lần trả `False` sớm đều có lý do chắc chắn: ký tự khác nhau hoặc `0` dẫn đầu.

Khi dừng, `abbr` là viết tắt của **toàn bộ** `word` **khi và chỉ khi** cả hai được dùng hết: `i == n and j == m`. Nếu `i > n` (nhảy quá), điều kiện sai → `False` đúng.

## 10. Độ phức tạp

Gọi `n = len(word)`, `m = len(abbr)`.

| Cách | Time | Space |
|---|:---:|:---:|
| Brute force sinh mọi viết tắt | `O(2^n · n)` | `O(2^n · n)` |
| Tách token | `O(m)` (+ so khớp `O(min(n, m))`) | `O(m)` cho danh sách token |
| **Hai con trỏ** | `O(m)` — mỗi ký tự `abbr` xử lý 1 lần; `i` nhảy theo số chứ không lặp từng ký tự | `O(1)` |
| Regex | phụ thuộc engine; thực tế `O(n + m)` | `O(m)` cho pattern |

Lưu ý: hai con trỏ **không** tốn `O(n)` dù `i` đi tới `n`, vì mỗi lần gặp số ta nhảy một phát. Tổng số vòng ≤ `m`.

## 11. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<38} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", "internationalization", "i12iz4n", True),
    ("đề 2", "apple", "a2e", False),
    ("s10n", "substitution", "s10n", True),
    ("sub4u4", "substitution", "sub4u4", True),
    ("toàn bộ là số 12", "substitution", "12", True),
    ("su3i1u2on", "substitution", "su3i1u2on", True),
    ("không viết tắt", "substitution", "substitution", True),
    ("s55n: kề nhau -> số 55", "substitution", "s55n", False),
    ("s010n: 0 dẫn đầu", "substitution", "s010n", False),
    ("s0ubstitution: đoạn rỗng", "substitution", "s0ubstitution", False),
    ("nhảy quá đuôi", "apple", "a5", False),
    ("abbr dư ký tự", "apple", "apple1", False),
    ("word dư ký tự", "apple", "app", False),
    ("1 ký tự = 1", "a", "1", True),
    ("số lớn 32-bit", "a", "2147483647", False),
    ("số đúng ở cuối", "hello", "h4", True),
]
for name, w, a, exp in cases:
    check(name, sol.validWordAbbreviation(w, a), exp)

PASS  đề 1                                   -> True
PASS  đề 2                                   -> False
PASS  s10n                                   -> True
PASS  sub4u4                                 -> True
PASS  toàn bộ là số 12                       -> True
PASS  su3i1u2on                              -> True
PASS  không viết tắt                         -> True
PASS  s55n: kề nhau -> số 55                 -> False
PASS  s010n: 0 dẫn đầu                       -> False
PASS  s0ubstitution: đoạn rỗng               -> False
PASS  nhảy quá đuôi                          -> False
PASS  abbr dư ký tự                          -> False
PASS  word dư ký tự                          -> False
PASS  1 ký tự = 1                            -> True
PASS  số lớn 32-bit                          -> False
PASS  số đúng ở cuối                         -> True


In [16]:
import random
rng = random.Random(408)
count = 0
for _ in range(300):
    n = rng.randint(1, 8)
    word = "".join(rng.choice("ab") for _ in range(n))
    valid = all_abbreviations(word)
    # thử cả viết tắt hợp lệ lẫn chuỗi ngẫu nhiên (có thể chứa 0)
    candidates = list(rng.sample(sorted(valid), min(5, len(valid))))
    candidates += ["".join(rng.choice("ab0123") for _ in range(rng.randint(1, 6))) for _ in range(5)]
    for abbr in candidates:
        exp = abbr in valid
        for f in (valid_abbr_two_pointers, valid_abbr_tokens, valid_abbr_regex, sol.validWordAbbreviation):
            assert f(word, abbr) == exp, (f.__name__, word, abbr)
        count += 1
print(f"{count} cặp (word, abbr): 4 cách khớp brute force -> PASS")

2823 cặp (word, abbr): 4 cách khớp brute force -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ đọc **một** chữ số
```python
if abbr[j].isdigit():
    i += int(abbr[j]); j += 1        # "12" bị đọc thành 1 rồi 2
```
**Vì sao sai:** `"12"` bị hiểu thành "nhảy 1 rồi nhảy 2" = nhảy 3, thay vì nhảy 12. Ngoài ra `"111"` thành nhảy 3 thay vì 111. **Sửa:** dùng vòng `while` đọc trọn cụm số.

In [17]:
def wrong_single_digit(word, abbr):
    i = j = 0
    while i < len(word) and j < len(abbr):
        if abbr[j].isdigit():
            i += int(abbr[j]); j += 1
        else:
            if word[i] != abbr[j]:
                return False
            i += 1; j += 1
    return i == len(word) and j == len(abbr)

w = "internationalization"
print("đọc 1 chữ số:", wrong_single_digit(w, "i12iz4n"), "| đúng:", sol.validWordAbbreviation(w, "i12iz4n"))
print("đọc 1 chữ số với 'abcd','a111':", wrong_single_digit("abcd", "a111"), "(đúng phải False vì '111' là 111)")

đọc 1 chữ số: False | đúng: True
đọc 1 chữ số với 'abcd','a111': True (đúng phải False vì '111' là 111)


### Lỗi 2: Quên chặn số 0 dẫn đầu
Không kiểm tra `abbr[j] == '0'` → `"s010n"` đọc thành 10 → `True` sai; `"s0ubstitution"` nhảy 0 → `True` sai.

### Lỗi 3: Chỉ kiểm `i == n` mà quên `j == m` (hoặc ngược lại)
`"apple"`, `"apple1"` → `i == n` đúng nhưng `abbr` còn dư.

In [18]:
def wrong_no_zero_check(word, abbr):
    i = j = 0
    while i < len(word) and j < len(abbr):
        if abbr[j].isdigit():
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j]); j += 1
            i += num
        else:
            if word[i] != abbr[j]:
                return False
            i += 1; j += 1
    return i == len(word)          # cũng quên j == m

print("s010n:", wrong_no_zero_check("substitution", "s010n"), "(đúng: False)")
print("s0ubstitution:", wrong_no_zero_check("substitution", "s0ubstitution"), "(đúng: False)")
print("apple1:", wrong_no_zero_check("apple", "apple1"), "(đúng: False)")

s010n: True (đúng: False)
s0ubstitution: True (đúng: False)
apple1: True (đúng: False)


### Lỗi 4: Đọc `word[i]` khi `i` đã vượt `len(word)`
Ở cách token, sau khi nhảy `i` có thể `>= len(word)`; `word[i]` gây `IndexError`. **Sửa:** kiểm `i >= len(word)` trước.

### Lỗi 5: Dùng `int(abbr[j:])` hay `split`
Cắt cả phần còn lại rồi `int(...)` sẽ lỗi vì phần sau có chữ cái. Phải dừng đúng ở chữ số cuối cùng.

In [19]:
try:
    print(int("12iz4n"))
except ValueError as e:
    print("ValueError:", e)

ValueError: invalid literal for int() with base 10: '12iz4n'


## 13. Mẹo nhận dạng pattern

- **Hai chuỗi so khớp theo luật** (một chuỗi là "mẫu nén" của chuỗi kia) → hai con trỏ, mỗi con trỏ tiến theo luật riêng.
- **Chuỗi có số nhiều chữ số lẫn chữ cái** → vòng `while ... isdigit()` + `num = num*10 + d`.
- Kết thúc luôn hỏi: "cả hai con trỏ đã **đi hết** chưa?"
- Các bài cùng họ: 443 String Compression, 394 Decode String, 2060 Check if an Original String Exists Given Two Encoded Strings (khó), 527/411 Word Abbreviation (Hard, Premium).

## 14. Biến thể

1. **Sinh mọi viết tắt** (LeetCode 320 Generalized Abbreviation) — chính là `all_abbreviations`.
2. **Viết tắt dạng chuẩn** `"i18n"`: chữ đầu + số ký tự giữa + chữ cuối.
3. **Mở rộng** một viết tắt thành mẫu với dấu `?` cho ký tự chưa biết: `"a2e"` → `"a??e"`.

In [20]:
def short_form(word: str) -> str:
    if len(word) <= 2:
        return word
    return word[0] + str(len(word) - 2) + word[-1]

def expand_abbr(abbr: str) -> str:
    out = []
    j = 0
    while j < len(abbr):
        if abbr[j].isdigit():
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j]); j += 1
            out.append("?" * num)
        else:
            out.append(abbr[j]); j += 1
    return "".join(out)

print(short_form("internationalization"), short_form("localization"), short_form("ok"))
print(expand_abbr("a2e"), expand_abbr("i12iz4n"))

i18n l10n ok
a??e i????????????iz????n


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** `word = "localization"`, `abbr = "l10n"`. Lập bảng `j`, `abbr[j]`, `i`. Kết quả?

**Bài 2 (đoán output):** `Solution().validWordAbbreviation("hi", "02")` và `("hi", "2")`?

**Bài 3 (điền chỗ trống):** hoàn thiện hai con trỏ.

In [21]:
def exercise_valid_abbr(word: str, abbr: str) -> bool:
    i = j = 0
    while i < len(word) and j < len(abbr):
        # TODO 1: nếu abbr[j] là chữ số: chặn '0', đọc trọn số, i += num
        # TODO 2: ngược lại: so ký tự, tiến cả hai
        pass
    # TODO 3: trả về điều kiện cả hai cùng hết
    pass

# assert exercise_valid_abbr("internationalization", "i12iz4n") is True
# assert exercise_valid_abbr("apple", "a2e") is False

**Bài 4 (tự viết):** `exercise_count_replaced(abbr)` trả **tổng số ký tự bị thay** (tổng các số) trong `abbr`. Ví dụ `"i12iz4n" → 16`.

**Bài 5 (biến thể):** `exercise_abbr_length(word, abbr)` trả độ dài từ gốc **suy ra từ `abbr`** (số chữ cái + tổng các số), rồi dùng nó để loại nhanh: nếu khác `len(word)` → `False` ngay. Viết hàm `exercise_quick_reject(word, abbr) -> bool` trả `True` nếu chắc chắn **không** hợp lệ theo độ dài.

**Bài 6 (mini challenge):** `exercise_matches_any(words, abbr)` trả list các từ trong `words` mà `abbr` là viết tắt hợp lệ.

In [22]:
def exercise_count_replaced(abbr: str) -> int:
    # TODO
    pass

def exercise_quick_reject(word: str, abbr: str) -> bool:
    # TODO
    pass

def exercise_matches_any(words: list, abbr: str) -> list:
    # TODO
    pass

# assert exercise_count_replaced("i12iz4n") == 16
# assert exercise_quick_reject("apple", "a5") is True
# assert exercise_matches_any(["apple", "angle", "ample", "apply"], "a3e") == ["apple", "angle", "ample"]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chữ → so, số → nhảy | `l` khớp, nhảy 10 | `word[11]` là gì? |
| 2 | Chữ số đầu là gì? | `'0'` → False | `"2"` nhảy 2 đúng bằng độ dài |
| 3 | Theo mục 5 | `while j < m and abbr[j].isdigit()` | `return i == len(word) and j == len(abbr)` |
| 4 | Cũng đọc cụm số | Cộng mỗi số đọc được | `total += num` sau vòng đọc số |
| 5 | Mỗi chữ cái = 1 ký tự | `length = letters + sum(numbers)` | `return length != len(word)` |
| 6 | Dùng lại hàm chính | List comprehension | `[w for w in words if valid(w, abbr)]` |

## 17. Đáp án tham khảo

- **Bài 1:** `j=0 'l'` khớp `word[0]`, `i=1`; `j=1..2 "10"` → `i = 11`; `j=3 'n'` so `word[11] = 'n'` khớp → `i = 12 = n`, `j = 4 = m` → **True**.
- **Bài 2:** `"02"` → gặp `'0'` → **False**; `"2"` → **True**.

In [23]:
print("Bài 1:", Solution().validWordAbbreviation("localization", "l10n"))
print("Bài 2:", Solution().validWordAbbreviation("hi", "02"), Solution().validWordAbbreviation("hi", "2"))

Bài 1: True
Bài 2: False True


In [24]:
def answer_valid_abbr(word: str, abbr: str) -> bool:
    i = j = 0
    while i < len(word) and j < len(abbr):
        if abbr[j].isdigit():
            if abbr[j] == "0":
                return False
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j]); j += 1
            i += num
        else:
            if word[i] != abbr[j]:
                return False
            i += 1; j += 1
    return i == len(word) and j == len(abbr)

def answer_count_replaced(abbr: str) -> int:
    total, num = 0, 0
    for ch in abbr:
        if ch.isdigit():
            num = num * 10 + int(ch)
        else:
            total += num
            num = 0
    return total + num            # số ở cuối chuỗi

def answer_quick_reject(word: str, abbr: str) -> bool:
    letters = sum(1 for ch in abbr if not ch.isdigit())
    return letters + answer_count_replaced(abbr) != len(word)

def answer_matches_any(words: list, abbr: str) -> list:
    return [w for w in words if answer_valid_abbr(w, abbr)]

assert answer_valid_abbr("internationalization", "i12iz4n") is True
assert answer_valid_abbr("apple", "a2e") is False
assert answer_count_replaced("i12iz4n") == 16 and answer_count_replaced("12") == 12
assert answer_quick_reject("apple", "a5") is True and answer_quick_reject("apple", "a3e") is False
assert answer_matches_any(["apple", "angle", "ample", "apply"], "a3e") == ["apple", "angle", "ample"]
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def validWordAbbreviation(self, word: str, abbr: str) -> bool:
        i = j = 0
        while i < len(word) and j < len(abbr):
            if abbr[j].isdigit():
                if abbr[j] == "0": return False          # 0 dẫn đầu / đoạn rỗng
                num = 0
                while j < len(abbr) and abbr[j].isdigit():
                    num = num * 10 + int(abbr[j]); j += 1
                i += num
            else:
                if word[i] != abbr[j]: return False
                i += 1; j += 1
        return i == len(word) and j == len(abbr)
```

**Checklist:**
- [ ] Đọc **trọn** cụm số nhiều chữ số.
- [ ] Chặn `'0'` ở đầu cụm số.
- [ ] Không đọc `word[i]` khi `i` đã ra ngoài.
- [ ] Kết thúc kiểm **cả** `i == len(word)` và `j == len(abbr)`.
- [ ] Time `O(m)`, Space `O(1)`.